In [1]:
# ── Cell 0: Setup (chạy đầu tiên) ──
import os

# Clone repo nếu chưa có
if not os.path.exists('/content/project'):
    !git clone https://github.com/TruongDuyLongPTIT/CTQW_PRO_METABOLITES_PRIORITIZING.git /content/project

# Add src vào Python path
import sys
sys.path.insert(0, '/content/project/src')

# Install dependencies nếu cần
!pip install -q torch scikit-learn networkx tqdm

Cloning into '/content/project'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 145 (delta 84), reused 56 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 1021.12 KiB | 9.12 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [2]:
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print("Drive đã mount sẵn.")

Mounted at /content/drive


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FINAL: PROFANCY vs CTQW-PRO vs NH-CTQW-PRO — degree vs RANK (Spearman)
# trên G_pro (pathway-augmented graph, Table 2/3 trong pipeline gốc)
# Paste toàn bộ vào 1 cell Colab
#
# Chỉ đo Spearman rho(degree, rank) — không tính Pearson trên score nữa.
# Cofactor KHÔNG bị loại khỏi graph — cả 3 method đều chạy trên G_pro đầy
# đủ, đúng tinh thần CTQW-PRO/NH-CTQW-PRO không cần loại cofactor thủ công.
# ═══════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/project/src')

import numpy as np
from scipy import stats
from scipy.stats import rankdata

from config import CACHE_DIR, T_FIXED, NH_GAMMA, RECON3D_COFACTORS, RESULTS_DIR
from graph import (parse_recon3d, build_gcc, build_gpro, compute_eigendecomp,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon)
from eval_sets import parse_hmdb, build_hmdb_lookups, build_eval_set3
from methods import make_profancy, make_ctqw_pro, make_nh_pro

N_DISEASES  = None   # None = full SMPDB
RANDOM_SEED = 42

# ── Setup: G_cc → G_pro (pathway-augmented) ──────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, _ = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)
print(f'  G_pro: {N_PRO} nodes ({N_PRO - N} pathway nodes), '
      f'{G_pro.number_of_edges()} edges')

deg_pro_safe = np.where(deg_pro > 0, deg_pro, 1.0)
P_pro        = A_pro / deg_pro_safe[:, None]

eigvals, eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

# Degree trong G_pro, map về không gian index G_cc bằng đúng _pro_src/_pro_dst
# mà methods.py dùng để map score — score[i] và degree[i] luôn khớp cùng node.
deg_by_gcc_idx = np.zeros(N)
deg_by_gcc_idx[_pro_dst] = deg_pro[_pro_src]

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])

eval_set3 = build_eval_set3(hmdb_metabolites, hmdb_to_recon, node_idx)
print(f'  SMPDB: {len(eval_set3)} diseases')

diseases_list = list(eval_set3.items())
if N_DISEASES is not None:
    rng = np.random.default_rng(RANDOM_SEED)
    idx = rng.choice(len(diseases_list), size=min(N_DISEASES, len(diseases_list)), replace=False)
    diseases_list = [diseases_list[i] for i in idx]
print(f'  Using {len(diseases_list)} diseases')

# ── Method closures — PROFANCY, CTQW-PRO, NH-CTQW-PRO trên G_pro ─────
run_profancy_fn = make_profancy(P_pro, idx_pro, node_idx, N, N_PRO)

_run_ctqw_pro   = make_ctqw_pro(eigvals, eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst)
run_ctqw_pro_fn = lambda seeds: _run_ctqw_pro(seeds, [T_FIXED])[T_FIXED]

print(f'  Building NH eigendecomp (gamma={NH_GAMMA})...')
run_nh_fn = make_nh_pro(
    A_pro, idx_pro, N, N_PRO, _pro_src, _pro_dst,
    RECON3D_COFACTORS, pro_nodes, NH_GAMMA, T_FIXED)

METHODS = {
    'PROFANCY':                       run_profancy_fn,
    'CTQW-PRO':                       run_ctqw_pro_fn,
    f'NH-CTQW-PRO (gamma={NH_GAMMA})': run_nh_fn,
}


def collect_rank_degree(method_fn):
    """
    Seeds = toàn bộ metabolite của bệnh (không LOO — không cần held-out
    label vì đây là câu hỏi "degree ảnh hưởng rank thế nào", không phải
    ranking accuracy). Candidate = mọi node còn lại (loại seed), cofactor
    KHÔNG bị loại — graph và candidate pool giữ nguyên đầy đủ.
    """
    ranks_all, degs_all = [], []
    for disease, mets in diseases_list:
        valid = [m for m in mets if m in node_idx]
        if len(valid) < 3:
            continue
        seeds    = valid
        seed_set = {node_idx[s] for s in seeds}

        scores = method_fn(seeds)

        mask = np.ones(N, dtype=bool)
        for si in seed_set: mask[si] = False

        cand_idx = np.where(mask)[0]
        if len(cand_idx) < 2 or scores[cand_idx].sum() < 1e-12:
            continue

        cand_scores = scores[cand_idx]
        cand_ranks  = rankdata(-cand_scores, method='average')  # rank 1 = score cao nhất

        ranks_all.append(cand_ranks)
        degs_all.append(deg_by_gcc_idx[cand_idx])

    if not ranks_all:
        return np.array([]), np.array([])
    return np.concatenate(ranks_all), np.concatenate(degs_all)


print('\n' + '='*60)
print('DEGREE vs RANK — Spearman correlation (G_pro, SMPDB)')
print('='*60)

results = []
for name, fn in METHODS.items():
    rank_arr, deg_arr = collect_rank_degree(fn)
    if len(rank_arr) == 0:
        print(f'  {name}: no data — skip')
        continue
    # -rank để rho dương lớn nghĩa là "degree cao -> rank tot (score cao)"
    rho, p = stats.spearmanr(deg_arr, -rank_arr)
    print(f'  {name:<28} rho = {rho:+.3f}  (p={p:.2e})  n = {len(rank_arr):,}')
    results.append({'method': name, 'spearman_rho': rho, 'p_value': p, 'n_pairs': len(rank_arr)})

print('\nSUMMARY')
for r in results:
    print(f"  {r['method']:<28} rho(degree,rank) = {r['spearman_rho']:+.3f}")

if len(results) >= 2:
    base = results[0]['spearman_rho']
    for r in results[1:]:
        print(f"  Gap PROFANCY - {r['method']}: {base - r['spearman_rho']:+.3f}")

# Save
import pandas as pd
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out = RESULTS_DIR / 'degree_rank_correlation_gpro.csv'
pd.DataFrame(results).to_csv(out, index=False)
print(f'\nSaved: {out}')
print('Done.')